# MEK oscillation IHC analysis

This notebook measures Her7-Venus, pMEK and ppERK along the PSM of each embryo and compares the profiles between oscillation phases. The calculations are in `ihc_pipeline.py`. Bands can be drawn in the napari editor (`ihc_roi_editor.py`) or in Fiji (`IHC_ROI_only_CSV.ijm`).

The sections follow the analysis: find the samples, prepare the bands, make masks and measure, build profiles, and plot the phases. It is worth running one sample first and checking its QC figure before running all of them.

The images are only read. Everything the notebook produces is saved in the output folder.

## 1. Setup

Install the environment once with `uv sync --locked` (adding `--extra cuda` on the NVIDIA machine) and choose the project's `.venv` as the kernel. The README has the details. The next cell expects to start in the `analysis_scripts` folder. If the kernel starts somewhere else, set `ANALYSIS_DIR` to that folder.

All settings are in the cell after that. The `settings` dictionary holds the common ones, and `ADVANCED` can set any other setting by name. The table below lists them all.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

ANALYSIS_DIR = Path.cwd()
if not (ANALYSIS_DIR / "ihc_pipeline.py").exists():
    raise FileNotFoundError("Set ANALYSIS_DIR to the project's analysis_scripts directory.")
sys.path.insert(0, str(ANALYSIS_DIR))
import ihc_pipeline as ihc

In [ ]:
# -------- Files and sample selection --------
# The folder that holds the replicate folders (N_1..., N_2...) and the folder for
# results. The environment variables, when set, replace these paths.
EXPERIMENTAL_DATA = Path(os.environ.get(
    "IHC_EXPERIMENTAL_DATA", str(ANALYSIS_DIR.parent / "Exp_1__mek_h7v_pperk")))
OUTPUT_ROOT = Path(os.environ.get("IHC_OUTPUT_ROOT", str(ANALYSIS_DIR / "ihc_results")))

# Optional table of distance ranges to leave out (see section 6); skipped if absent.
EXCLUSIONS = ANALYSIS_DIR / "IHC_exclusions.csv"

# Samples to run, as "replicate/sample_id". Empty runs every sample that has ROIs.
SAMPLE_KEYS = []

# -------- What to run --------
RUN_MEASUREMENTS = True     # make masks and measure the selected samples
RELOAD_SAVED = True         # otherwise, load the saved measurements
OPEN_EDITOR = False         # open the napari ROI and mask editor in its own window

# -------- Settings (the table below lists all of them) --------
settings = dict(
    experimental_data=EXPERIMENTAL_DATA,
    output_root=OUTPUT_ROOT,
    replicates=(),                    # e.g. ("N_1", "N_2"); empty uses all
    nis_preference="nd2",             # denoised ND2 or TIFF, when both exist
    use_ai_denoise=True,              # segment the NIS-denoised DAPI
    use_cellpose=True,                # False thresholds the DAPI instead
    mask_source="compute",           # compute, existing (e.g. Fiji) or edited (napari)
    width_override_px=None,          # None uses each ROI's own width
    combine_sides=False,             # True combines an embryo's sides
    analysis_method="per_slice",     # per_slice, pooled, block_mip or full_mip
    z_average="pool",                # pool, mean or median over Z
    background_subtract=True,
    min_mask_fraction=0.05,          # blank the mean where a compartment covers < 5% of the band
    protein_median_size=3,           # 0 turns the median filter off
    protein_outliers=True,           # replace bright outlier pixels
    mask_cap=4095.0,                 # None turns the cap off
    exclusions_csv=str(EXCLUSIONS if EXCLUSIONS.exists() else ""),  # ranges to leave out
    device="auto",                  # CUDA, then Apple MPS, then CPU
    image_workers=6,                 # samples measured in parallel
    threads_per_worker=1,
    phase_files={},                  # e.g. {"N_1": "/path/phases.csv"}
    phase_overrides={},              # e.g. {"N_2/triple_ihc_014_P1": "2"}
)

# Any other setting, by its name in the table below.
ADVANCED = {
    # "cellprob_threshold": 0.0,
    # "flow_threshold": 0.4,
    # "cellpose_diameter_px": None,
    # "normalization_window_um": 40.0,
    # "smooth_average": True,          # smooths the plotted average only
    "phase_colormap": "Dark2",
    # --- baseline and day normalization ---
    "background_percentile": 5.0,          # baseline = this percentile of the smoothed profiles
    "normalization_window_um": 50.0,       # rolling-median width used to find the baseline
    "day_normalize": True,                 # divide each imaging day by its median, per channel and compartment
    "day_reference_um": (0.0, 400.0),      # distance range (µm) that sets each day's median

    # --- Savitzky–Golay smoothing of the plotted average ---
    "smooth_average": True,                # False = no smoothing
    "savgol_window_um":40.0,              # window width (µm)
    "savgol_order": 1,                     # polynomial order
    "interpolation": "nearest",
    "mip_block_size": 5,
    "phase_minimum_lines": 2,
}
settings.update(ADVANCED)
cfg = ihc.Config(**settings).validate()

### Settings reference

This table lists every setting with its current value, its allowed choices and a description. `None` means a setting is off, or that its value is read from the image; the description says which.

With `use_ai_denoise=True`, an NIS-denoised export is segmented as it is, without further filtering.

In [ ]:
parameters = ihc.parameter_table(cfg)
with pd.option_context("display.max_colwidth", 140, "display.max_rows", None):
    display(parameters)

## 2. Samples, NIS pairing and phases

Each row below is one sample: an exported TIFF, or one position of an ND2 file. When the same position exists as both, the TIFF is used for the protein channels. `nis_path` shows the NIS-denoised file whose DAPI will be segmented. When both a denoised ND2 and a denoised TIFF exist, `nis_preference` decides between them.

Phases come from `phases.csv` in each replicate folder, or from the file named in `phase_files`. The table can have the columns `file,phase` or `sample_id,phase`. `phase_source` shows which file each phase came from. Samples without a phase are still measured, and are only left out of the phase averages.

In [ ]:
samples = ihc.discover_samples(cfg)
samples["sample_key"] = samples.replicate + "/" + samples.sample_id
samples["has_nis"] = samples.nis_path.ne("")
display(samples[["sample_key", "has_rois", "has_nis", "phase", "phase_source", "raw_path", "nis_path"]])
display(samples.groupby("replicate").agg(
    images=("sample_id", "size"), ready_rois=("has_rois", "sum"), nis_pairs=("has_nis", "sum")))

selected = samples[samples.has_rois].copy()
if SAMPLE_KEYS:
    missing = set(SAMPLE_KEYS) - set(samples.sample_key)
    if missing:
        raise ValueError(f"Selected keys were not found: {sorted(missing)}")
    selected = selected[selected.sample_key.isin(SAMPLE_KEYS)].copy()
print(f"Selected {len(selected)} images with ROIs; {sum(~samples.has_rois)} images still need drawing.")

## 3. Bands (ROIs)

All the bands of an image are saved in one file, `output_root/rois/<replicate>/<sample>/rois.csv`. Each row is a clicked point (`roi_id, vertex_index, x_px, y_px, width_px`) in image pixels counted from 0, and the first point of a side is its posterior end. `width_px` is the width of the measured band, which is separate from how thick the line looks on screen.

The next cell converts ImageJ ROI ZIPs into this format. A CSV that already exists is kept, so running the cell again does not undo edits. Fiji (with `IHC_ROI_only_CSV.ijm`) and the napari editor both read and write these files.

The editor can also correct the nuclear mask. Corrected masks are saved under `edited_masks/` and are used when `mask_source="edited"`.

In [ ]:
conversion = ihc.convert_rois(samples[samples.has_rois], cfg)
display(conversion)
if OPEN_EDITOR:
    # Pass every sample, so bands can be drawn for samples that have none yet.
    editor_process = ihc.launch_editor(samples, cfg)
    print("The editor is opening in its own window. Save the ROIs there before measuring.")

## 4. Masks and measurements

With `use_ai_denoise=True`, the DAPI of each sample's NIS-denoised export is segmented. With `use_ai_denoise=False`, the raw DAPI is median filtered, background subtracted and contrast enhanced first. With `use_cellpose=False`, each Z-plane is thresholded instead of being segmented by Cellpose.

Every Z method is measured with the same filtered protein images, masks and bands. The cytoplasmic mask is everything that is not nucleus, and maximum projections are taken after masking. With 31 planes, the default blocks are 2–4, 5–7, …, 29–31.

Before measuring, the protein channels are median filtered, bright outlier pixels are replaced, and values inside the masks are capped at 4095. Each of these steps can be turned off in the settings.

Each sample's measurements are saved as Parquet (`export_csv=True` also writes CSV). Saved masks are reused while `reuse_cache` is True, so set it to False after changing a mask setting.

In [ ]:
measurements = pd.DataFrame()
if RUN_MEASUREMENTS:
    if selected.empty:
        raise ValueError("No selected sample has ROIs. Draw or convert ROIs first.")
    measurements = ihc.run(selected, cfg)
    print("Run metadata:", measurements.attrs.get("run_directory"))
elif RELOAD_SAVED:
    measurements = ihc.load_measurements(selected, cfg)
    print("Loaded the saved measurements.")
else:
    print("Nothing was measured or loaded. Set RUN_MEASUREMENTS or RELOAD_SAVED to True.")
if not measurements.empty:
    display(measurements.head())
    display(measurements.groupby(["replicate", "sample_id", "method"]).size().rename("rows"))

### What a profile contains

At each position, the band has many points across its width, and `mask_weight / band_weight` is the fraction of them inside the compartment. A point at the edge of a nucleus can count partly, because the mask is interpolated.

- `masked_mean = masked_sum / mask_weight` is the mean intensity inside the compartment.
- `band_mean = masked_sum / band_weight` is the compartment's signal per band area.
- Where the compartment covers less than `min_mask_fraction` of the band, `masked_mean` is left blank. The sums are kept, so pooling over Z still uses them.

A pooled profile adds the sums of all planes before dividing, which weights each plane by how much compartment it contains. Averaging the per-plane means would weight the planes equally instead.

## 5. Check a mask

The figure shows the raw DAPI of the first selected sample, and beside it the same plane with the nuclear mask in green, the measured bands in blue and the posterior start of each band in red. It uses the same mask as the measurements. It is worth looking at a few Z-planes (the `z` argument of `qc_figure`) and checking that a mask made from NIS-denoised DAPI lines up with the raw image.

In [ ]:
if not measurements.empty:
    figure = ihc.qc_figure(selected.iloc[0], cfg)
    display(figure)
    plt.close(figure)

## 6. Profiles

Ranges listed in the exclusions CSV are left out here. Its columns are `replicate,sample_id,side,start_um,stop_um,channels,reason`, with `all` in `channels` for every channel. The saved measurements are not changed.

Each embryo's baseline is a low percentile (`background_percentile`) of its profiles after smoothing them with a rolling median (`normalization_window_um`). It is shared by all sides of the embryo, computed separately for each channel and compartment, and subtracted from the unsmoothed profiles. With `day_normalize=True`, each imaging day is then divided by its median within `day_reference_um`.

By default each side is its own profile, and `combine_sides=True` combines an embryo's sides. Profiles end where their band ends. Distances are the position along the band times the pixel size.

In [ ]:
profiles = normalization = pd.DataFrame()
if not measurements.empty:
    profiles, normalization = ihc.side_profiles(measurements, cfg)
    display(normalization)
    display(profiles.head())
    individual_figure = ihc.plot_individual_profiles(profiles, cfg)
    display(individual_figure)
    plt.close(individual_figure)
    bad = normalization[normalization.status.ne("ok")]
    if len(bad):
        print("These embryos have no baseline because their profiles are empty:")
        display(bad)

## 7. Phase summaries

Phases come only from the annotation files. Labels not listed in `phase_labels` stay in the individual profiles but are left out of the phase plots.

With sides kept separate, `n` counts sides, and `n_embryos` and `n_replicates` are given alongside. With `combine_sides=True`, `n` counts embryos. `n` falls where profiles end or ranges are excluded, and an average is drawn only where `n` reaches `phase_minimum_lines`. The plotted average is smoothed when `smooth_average` is True; the saved statistics and error bands are not.

In [ ]:
phase_summary = overall_summary = pd.DataFrame()
if not profiles.empty:
    phase_summary = ihc.summarize(profiles, cfg, by_phase=True)
    overall_summary = ihc.summarize(profiles, cfg, by_phase=False)
    display(profiles.groupby(["replicate", "phase"]).profile_unit.nunique().rename("profile_units"))
    display(phase_summary.head())
    for plotter in (ihc.plot_phases, ihc.plot_phase_panels):
        for channel, figure in plotter(profiles, cfg, phase_summary).items():
            display(figure)
            plt.close(figure)

## 8. Save the analysis

This saves the profiles, baselines and day factors, the phase and overall summaries, the settings, the package versions and the figures in a new folder under `analysis/`.

To change only the normalization, side combination or plots, reload the saved measurements and rerun sections 6 to 8. If the phase annotations change, rerun section 2 first and then reload. Changes to masks, bands, protein filtering or Z settings need section 4 run again.

In [ ]:
if not profiles.empty:
    analysis_dir = ihc.export_analysis(profiles, normalization, phase_summary, cfg)
    overall_summary.to_csv(analysis_dir / "overall_summary.csv", index=False)
    individual_figure = ihc.plot_individual_profiles(profiles, cfg)
    individual_figure.savefig(analysis_dir / "individual_profiles.png", dpi=cfg.dpi)
    plt.close(individual_figure)
    for channel, figure in ihc.plot_phases(profiles, cfg, phase_summary).items():
        figure.savefig(analysis_dir / f"{channel}_phases.png", dpi=cfg.dpi)
        plt.close(figure)
    print("Saved:", analysis_dir)

## 9. Further exploration

`measurements`, `profiles`, `normalization`, `phase_summary` and `overall_summary` are ordinary pandas tables that can be explored here.